### setup

In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-01-27 03:05:13.397212: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/shevkunov/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


matrix_approx_zeshel.py


In [2]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [3]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances


def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [4]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        return ma.cur.latent_cols.T

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [5]:
L = 7000
N = 1000
DA = 50

# ctx = load(L, raw_path = "//dev/null/stand/log.local.txt", path="log.local.bin.old", seed=17, det_attempts=DA)
# .set_top_games_as_key()
# print("LOADED")

L = 7000
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6095219944863732e-120
2.6095219944863732e-120
101 4769 2088


In [6]:
import json

with open("games-all.json", "r") as f:
    j = json.loads(f.read())

In [7]:
idx2app = [x[0] for x in ctx.get_requests("train")[0][2]]

app2cat = dict()

for j_i in j:
    app2cat[j_i["appID"]] = j_i["categoriesNames"]


idx2cat = [
    (app2cat[a] if a in app2cat else []) 
    for a in idx2app
]

np.mean([(len(c) > 0) for c in idx2cat])

0.9908562431875984

In [8]:
C = list()
for c in idx2cat:
    for c_i in c:
        C.append(c_i)
C = sorted(list(set(C)))

In [9]:
Ch = {n:i for i, n in enumerate(C)}
Ch

{'action': 0,
 'adventure': 1,
 'applications': 2,
 'arcade': 3,
 'balloons': 4,
 'cards': 5,
 'casino': 6,
 'casual': 7,
 'economic': 8,
 'editors_choice': 9,
 'educational': 10,
 'for_babies': 11,
 'for_boys': 12,
 'for_girls': 13,
 'for_two_persons': 14,
 'games_io': 15,
 'horrors': 16,
 'imitations': 17,
 'kids': 18,
 'match3': 19,
 'midcore': 20,
 'puzzles': 21,
 'quiz': 22,
 'race': 23,
 'role': 24,
 'simulator': 25,
 'sports': 26,
 'strategy': 27,
 'tabletop': 28,
 'tests': 29}

In [10]:
T = np.array([
    ([1. * (n in row) for n in C] if row else [1,] + [0] * 29)
    for row in idx2cat
])

In [11]:
idx = np.arange(T.shape[0])
np.random.seed(17)
np.random.shuffle(idx)

In [12]:
import tensorflow as tf

In [13]:
def fit(GE, T, E=200):
    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Dense(100, input_dim = GE.shape[1], activation = 'elu')) # input layer requires input_dim param
    model.add(tf.keras.layers.Dense(50, activation = 'elu'))
    model.add(tf.keras.layers.Dense(30, activation = 'softmax'))

    loss = tf.keras.losses.CategoricalCrossentropy(
        from_logits=True
    )

    model.compile(loss='binary_crossentropy', optimizer= "adam", metrics=['accuracy'])

    #es = EarlyStopping(monitor='loss', min_delta=0.005, patience=1, verbose=1, mode='auto')
    model.fit(GE, T, epochs = E, shuffle = True, batch_size=128, verbose=2, validation_split=0.1)

    scores = model.evaluate(GE, T)
    print(model.metrics_names[0], model.metrics_names[1])
    
    return model


def getQ(model, GE, T, th):
    P_ = model.predict(GE)
    P = 1 * (P_ > th)

    return ((P * T).sum(axis=1) / np.clip(T + P, 0, 1).sum(axis=1)).mean()
    

In [14]:
ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
print(ma.get_score("train"), ma.get_score("test"))

GE = ma.get_game_embs().numpy()
GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

np.mean(results), mse, len(results) =  0.596399664499895 0.17181198363248248 4769
np.mean(results), mse, len(results) =  0.5867624521072797 0.18987861028145928 2088
0.596399664499895 0.5867624521072797
Epoch 1/200
93/93 - 2s - loss: 0.3418 - accuracy: 0.3148 - val_loss: 0.1117 - val_accuracy: 0.4887 - 2s/epoch - 17ms/step
Epoch 2/200
93/93 - 0s - loss: 0.0823 - accuracy: 0.5230 - val_loss: 0.0673 - val_accuracy: 0.5401 - 288ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.0550 - accuracy: 0.5409 - val_loss: 0.0509 - val_accuracy: 0.5424 - 285ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0431 - accuracy: 0.5515 - val_loss: 0.0428 - val_accuracy: 0.5439 - 283ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0364 - accuracy: 0.5545 - val_loss: 0.0381 - val_accuracy: 0.5522 - 278ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0321 - accuracy: 0.5583 - val_loss: 0.0351 - val_accuracy: 0.5499 - 280ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0290 - accuracy: 0.5632

93/93 - 0s - loss: 0.0039 - accuracy: 0.5902 - val_loss: 0.0387 - val_accuracy: 0.5756 - 282ms/epoch - 3ms/step
Epoch 66/200
93/93 - 0s - loss: 0.0037 - accuracy: 0.5958 - val_loss: 0.0395 - val_accuracy: 0.5726 - 288ms/epoch - 3ms/step
Epoch 67/200
93/93 - 0s - loss: 0.0037 - accuracy: 0.5923 - val_loss: 0.0402 - val_accuracy: 0.5613 - 270ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0037 - accuracy: 0.5903 - val_loss: 0.0400 - val_accuracy: 0.5651 - 272ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0035 - accuracy: 0.5953 - val_loss: 0.0417 - val_accuracy: 0.5628 - 276ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0033 - accuracy: 0.5937 - val_loss: 0.0416 - val_accuracy: 0.5469 - 272ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0033 - accuracy: 0.5962 - val_loss: 0.0413 - val_accuracy: 0.5658 - 278ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0030 - accuracy: 0.5990 - val_loss: 0.0420 - val_accuracy: 0.5673 - 274ms/epoch - 3ms/step
Epoch 73/200


Epoch 130/200
93/93 - 0s - loss: 1.9892e-04 - accuracy: 0.6117 - val_loss: 0.0724 - val_accuracy: 0.5764 - 277ms/epoch - 3ms/step
Epoch 131/200
93/93 - 0s - loss: 1.8673e-04 - accuracy: 0.6119 - val_loss: 0.0727 - val_accuracy: 0.5772 - 269ms/epoch - 3ms/step
Epoch 132/200
93/93 - 0s - loss: 1.8024e-04 - accuracy: 0.6128 - val_loss: 0.0728 - val_accuracy: 0.5764 - 270ms/epoch - 3ms/step
Epoch 133/200
93/93 - 0s - loss: 1.7208e-04 - accuracy: 0.6122 - val_loss: 0.0732 - val_accuracy: 0.5734 - 272ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 1.6601e-04 - accuracy: 0.6110 - val_loss: 0.0735 - val_accuracy: 0.5719 - 270ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 1.5932e-04 - accuracy: 0.6115 - val_loss: 0.0736 - val_accuracy: 0.5772 - 272ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 1.5493e-04 - accuracy: 0.6100 - val_loss: 0.0741 - val_accuracy: 0.5734 - 268ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 1.4917e-04 - accuracy: 0.6111 - val_loss: 0.0742 - val_ac

Epoch 194/200
93/93 - 0s - loss: 4.9530e-05 - accuracy: 0.6271 - val_loss: 0.0919 - val_accuracy: 0.5877 - 283ms/epoch - 3ms/step
Epoch 195/200
93/93 - 0s - loss: 4.7245e-05 - accuracy: 0.6239 - val_loss: 0.0919 - val_accuracy: 0.5840 - 280ms/epoch - 3ms/step
Epoch 196/200
93/93 - 0s - loss: 4.5576e-05 - accuracy: 0.6242 - val_loss: 0.0919 - val_accuracy: 0.5855 - 294ms/epoch - 3ms/step
Epoch 197/200
93/93 - 0s - loss: 4.3775e-05 - accuracy: 0.6249 - val_loss: 0.0921 - val_accuracy: 0.5847 - 289ms/epoch - 3ms/step
Epoch 198/200
93/93 - 0s - loss: 4.2189e-05 - accuracy: 0.6252 - val_loss: 0.0922 - val_accuracy: 0.5847 - 275ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 4.1134e-05 - accuracy: 0.6255 - val_loss: 0.0923 - val_accuracy: 0.5809 - 284ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 3.9626e-05 - accuracy: 0.6230 - val_loss: 0.0923 - val_accuracy: 0.5855 - 282ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0093 - accuracy: 0.6219
l

In [25]:
np.random.seed(17)
ma = AnnCUR(ctx, key_games=np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False))

ma.fit()
print(ma.get_score("train"), ma.get_score("test"))

GE = ma.get_game_embs().numpy()
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

np.mean(results), mse, len(results) =  0.611203606626127 0.16897871993289876 4769
np.mean(results), mse, len(results) =  0.60330938697318 0.18623048026566658 2088
0.611203606626127 0.60330938697318
Epoch 1/200
93/93 - 1s - loss: 0.3737 - accuracy: 0.1680 - val_loss: 0.1716 - val_accuracy: 0.2315 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1569 - accuracy: 0.3163 - val_loss: 0.1399 - val_accuracy: 0.3623 - 280ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1248 - accuracy: 0.3934 - val_loss: 0.1094 - val_accuracy: 0.4236 - 279ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0973 - accuracy: 0.4248 - val_loss: 0.0867 - val_accuracy: 0.4380 - 276ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0781 - accuracy: 0.4572 - val_loss: 0.0718 - val_accuracy: 0.4675 - 277ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0656 - accuracy: 0.4873 - val_loss: 0.0618 - val_accuracy: 0.4720 - 277ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0572 - accuracy: 0.5006 - v

93/93 - 0s - loss: 0.0168 - accuracy: 0.5667 - val_loss: 0.0255 - val_accuracy: 0.5310 - 272ms/epoch - 3ms/step
Epoch 66/200
93/93 - 0s - loss: 0.0166 - accuracy: 0.5639 - val_loss: 0.0258 - val_accuracy: 0.5439 - 269ms/epoch - 3ms/step
Epoch 67/200
93/93 - 0s - loss: 0.0165 - accuracy: 0.5709 - val_loss: 0.0254 - val_accuracy: 0.5356 - 271ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0164 - accuracy: 0.5630 - val_loss: 0.0255 - val_accuracy: 0.5363 - 274ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0162 - accuracy: 0.5678 - val_loss: 0.0257 - val_accuracy: 0.5325 - 265ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0161 - accuracy: 0.5652 - val_loss: 0.0257 - val_accuracy: 0.5439 - 266ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0160 - accuracy: 0.5697 - val_loss: 0.0257 - val_accuracy: 0.5325 - 272ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0158 - accuracy: 0.5678 - val_loss: 0.0258 - val_accuracy: 0.5393 - 282ms/epoch - 3ms/step
Epoch 73/200


Epoch 131/200
93/93 - 0s - loss: 0.0106 - accuracy: 0.5651 - val_loss: 0.0277 - val_accuracy: 0.5371 - 277ms/epoch - 3ms/step
Epoch 132/200
93/93 - 0s - loss: 0.0105 - accuracy: 0.5631 - val_loss: 0.0279 - val_accuracy: 0.5424 - 280ms/epoch - 3ms/step
Epoch 133/200
93/93 - 0s - loss: 0.0105 - accuracy: 0.5763 - val_loss: 0.0278 - val_accuracy: 0.5424 - 274ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0104 - accuracy: 0.5660 - val_loss: 0.0278 - val_accuracy: 0.5446 - 272ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0104 - accuracy: 0.5709 - val_loss: 0.0282 - val_accuracy: 0.5477 - 275ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0103 - accuracy: 0.5677 - val_loss: 0.0278 - val_accuracy: 0.5424 - 270ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0102 - accuracy: 0.5699 - val_loss: 0.0282 - val_accuracy: 0.5499 - 268ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0101 - accuracy: 0.5712 - val_loss: 0.0282 - val_accuracy: 0.5416 - 273ms/epoch - 3

93/93 - 0s - loss: 0.0070 - accuracy: 0.5668 - val_loss: 0.0334 - val_accuracy: 0.5552 - 280ms/epoch - 3ms/step
Epoch 197/200
93/93 - 0s - loss: 0.0069 - accuracy: 0.5665 - val_loss: 0.0336 - val_accuracy: 0.5605 - 279ms/epoch - 3ms/step
Epoch 198/200
93/93 - 0s - loss: 0.0069 - accuracy: 0.5682 - val_loss: 0.0335 - val_accuracy: 0.5522 - 279ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0068 - accuracy: 0.5683 - val_loss: 0.0342 - val_accuracy: 0.5469 - 270ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0067 - accuracy: 0.5667 - val_loss: 0.0340 - val_accuracy: 0.5499 - 281ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0091 - accuracy: 0.5626
loss accuracy
104/104 [==============================] - 0s 2ms/step
0.7331214047835302


In [15]:
ctx.set_kmeans_games_as_key(all_from_labels=True)

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
print(ma.get_score("train"), ma.get_score("test"))

GE = ma.get_game_embs().numpy()
GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

np.mean(results), mse, len(results) =  0.6287607464877334 0.15658457174869786 4769
np.mean(results), mse, len(results) =  0.6183620689655172 0.17375032907563612 2088
0.6287607464877334 0.6183620689655172
Epoch 1/200
93/93 - 1s - loss: 0.3550 - accuracy: 0.3031 - val_loss: 0.1163 - val_accuracy: 0.4622 - 1s/epoch - 14ms/step
Epoch 2/200
93/93 - 0s - loss: 0.0863 - accuracy: 0.4948 - val_loss: 0.0693 - val_accuracy: 0.4909 - 271ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.0582 - accuracy: 0.5104 - val_loss: 0.0533 - val_accuracy: 0.5030 - 268ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0461 - accuracy: 0.5151 - val_loss: 0.0453 - val_accuracy: 0.5061 - 281ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0392 - accuracy: 0.5275 - val_loss: 0.0404 - val_accuracy: 0.5182 - 264ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0347 - accuracy: 0.5286 - val_loss: 0.0376 - val_accuracy: 0.5340 - 277ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0317 - accuracy: 0.53

Epoch 65/200
93/93 - 0s - loss: 0.0054 - accuracy: 0.5990 - val_loss: 0.0470 - val_accuracy: 0.5847 - 268ms/epoch - 3ms/step
Epoch 66/200
93/93 - 0s - loss: 0.0052 - accuracy: 0.6001 - val_loss: 0.0472 - val_accuracy: 0.5756 - 270ms/epoch - 3ms/step
Epoch 67/200
93/93 - 0s - loss: 0.0051 - accuracy: 0.6028 - val_loss: 0.0483 - val_accuracy: 0.5741 - 272ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0050 - accuracy: 0.6010 - val_loss: 0.0493 - val_accuracy: 0.5809 - 268ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0048 - accuracy: 0.6034 - val_loss: 0.0510 - val_accuracy: 0.5968 - 270ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0046 - accuracy: 0.6050 - val_loss: 0.0505 - val_accuracy: 0.5870 - 275ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0045 - accuracy: 0.5990 - val_loss: 0.0507 - val_accuracy: 0.5794 - 273ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0043 - accuracy: 0.6048 - val_loss: 0.0521 - val_accuracy: 0.5870 - 271ms/epoch - 3ms/step


Epoch 130/200
93/93 - 0s - loss: 0.0023 - accuracy: 0.6194 - val_loss: 0.0885 - val_accuracy: 0.5908 - 272ms/epoch - 3ms/step
Epoch 131/200
93/93 - 0s - loss: 9.9559e-04 - accuracy: 0.6186 - val_loss: 0.0880 - val_accuracy: 0.5847 - 271ms/epoch - 3ms/step
Epoch 132/200
93/93 - 0s - loss: 5.9202e-04 - accuracy: 0.6182 - val_loss: 0.0870 - val_accuracy: 0.5862 - 263ms/epoch - 3ms/step
Epoch 133/200
93/93 - 0s - loss: 4.0163e-04 - accuracy: 0.6212 - val_loss: 0.0879 - val_accuracy: 0.5893 - 266ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 3.4868e-04 - accuracy: 0.6212 - val_loss: 0.0885 - val_accuracy: 0.5893 - 267ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 3.2719e-04 - accuracy: 0.6241 - val_loss: 0.0888 - val_accuracy: 0.5900 - 273ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 3.0849e-04 - accuracy: 0.6221 - val_loss: 0.0892 - val_accuracy: 0.5908 - 275ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 2.9003e-04 - accuracy: 0.6218 - val_loss: 0.0896 - val_accura

Epoch 194/200
93/93 - 0s - loss: 6.9202e-05 - accuracy: 0.6264 - val_loss: 0.1124 - val_accuracy: 0.5983 - 273ms/epoch - 3ms/step
Epoch 195/200
93/93 - 0s - loss: 6.7506e-05 - accuracy: 0.6263 - val_loss: 0.1127 - val_accuracy: 0.6014 - 266ms/epoch - 3ms/step
Epoch 196/200
93/93 - 0s - loss: 6.5751e-05 - accuracy: 0.6264 - val_loss: 0.1128 - val_accuracy: 0.6021 - 266ms/epoch - 3ms/step
Epoch 197/200
93/93 - 0s - loss: 6.3888e-05 - accuracy: 0.6272 - val_loss: 0.1130 - val_accuracy: 0.6006 - 278ms/epoch - 3ms/step
Epoch 198/200
93/93 - 0s - loss: 6.2138e-05 - accuracy: 0.6276 - val_loss: 0.1132 - val_accuracy: 0.6029 - 274ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 6.0954e-05 - accuracy: 0.6281 - val_loss: 0.1132 - val_accuracy: 0.6021 - 275ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 5.9464e-05 - accuracy: 0.6284 - val_loss: 0.1136 - val_accuracy: 0.6059 - 269ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0114 - accuracy: 0.6276
l

In [24]:
ctx.set_kmeans_games_as_key(all_from_labels=True)

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
print(ma.get_score("train"), ma.get_score("test"))

GE = ma.get_game_embs().numpy()
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

np.mean(results), mse, len(results) =  0.6287607464877334 0.15658457174869786 4769
np.mean(results), mse, len(results) =  0.6183620689655172 0.17375032907563612 2088
0.6287607464877334 0.6183620689655172
Epoch 1/200
93/93 - 2s - loss: 0.3656 - accuracy: 0.2410 - val_loss: 0.1696 - val_accuracy: 0.3207 - 2s/epoch - 16ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1574 - accuracy: 0.3423 - val_loss: 0.1407 - val_accuracy: 0.3638 - 287ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1243 - accuracy: 0.4164 - val_loss: 0.1085 - val_accuracy: 0.4508 - 287ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0979 - accuracy: 0.4541 - val_loss: 0.0883 - val_accuracy: 0.4713 - 336ms/epoch - 4ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0809 - accuracy: 0.4770 - val_loss: 0.0744 - val_accuracy: 0.4924 - 287ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0689 - accuracy: 0.4977 - val_loss: 0.0647 - val_accuracy: 0.5068 - 275ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0604 - accuracy: 0.50

Epoch 65/200
93/93 - 0s - loss: 0.0202 - accuracy: 0.5668 - val_loss: 0.0286 - val_accuracy: 0.5537 - 284ms/epoch - 3ms/step
Epoch 66/200
93/93 - 0s - loss: 0.0201 - accuracy: 0.5628 - val_loss: 0.0287 - val_accuracy: 0.5545 - 277ms/epoch - 3ms/step
Epoch 67/200
93/93 - 0s - loss: 0.0199 - accuracy: 0.5619 - val_loss: 0.0289 - val_accuracy: 0.5560 - 274ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0198 - accuracy: 0.5624 - val_loss: 0.0287 - val_accuracy: 0.5461 - 281ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0196 - accuracy: 0.5625 - val_loss: 0.0287 - val_accuracy: 0.5605 - 283ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0195 - accuracy: 0.5696 - val_loss: 0.0289 - val_accuracy: 0.5499 - 279ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0194 - accuracy: 0.5638 - val_loss: 0.0287 - val_accuracy: 0.5590 - 280ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0193 - accuracy: 0.5609 - val_loss: 0.0287 - val_accuracy: 0.5567 - 282ms/epoch - 3ms/step


Epoch 131/200
93/93 - 0s - loss: 0.0135 - accuracy: 0.5768 - val_loss: 0.0319 - val_accuracy: 0.5530 - 281ms/epoch - 3ms/step
Epoch 132/200
93/93 - 0s - loss: 0.0134 - accuracy: 0.5752 - val_loss: 0.0319 - val_accuracy: 0.5643 - 286ms/epoch - 3ms/step
Epoch 133/200
93/93 - 0s - loss: 0.0133 - accuracy: 0.5778 - val_loss: 0.0316 - val_accuracy: 0.5688 - 288ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0132 - accuracy: 0.5757 - val_loss: 0.0318 - val_accuracy: 0.5658 - 297ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0132 - accuracy: 0.5785 - val_loss: 0.0317 - val_accuracy: 0.5651 - 293ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0131 - accuracy: 0.5776 - val_loss: 0.0325 - val_accuracy: 0.5756 - 282ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0130 - accuracy: 0.5754 - val_loss: 0.0322 - val_accuracy: 0.5620 - 281ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0129 - accuracy: 0.5757 - val_loss: 0.0321 - val_accuracy: 0.5620 - 285ms/epoch - 3

93/93 - 0s - loss: 0.0093 - accuracy: 0.5789 - val_loss: 0.0379 - val_accuracy: 0.5681 - 279ms/epoch - 3ms/step
Epoch 197/200
93/93 - 0s - loss: 0.0093 - accuracy: 0.5815 - val_loss: 0.0381 - val_accuracy: 0.5802 - 280ms/epoch - 3ms/step
Epoch 198/200
93/93 - 0s - loss: 0.0092 - accuracy: 0.5834 - val_loss: 0.0381 - val_accuracy: 0.5643 - 278ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0091 - accuracy: 0.5777 - val_loss: 0.0384 - val_accuracy: 0.5688 - 283ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0091 - accuracy: 0.5826 - val_loss: 0.0384 - val_accuracy: 0.5605 - 278ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0116 - accuracy: 0.5724
loss accuracy
104/104 [==============================] - 0s 1ms/step
0.7418609345039863


In [16]:
GE = ctx.docblocks[6]
GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.3074 - accuracy: 0.3428 - val_loss: 0.1025 - val_accuracy: 0.4682 - 1s/epoch - 14ms/step
Epoch 2/200
93/93 - 0s - loss: 0.0873 - accuracy: 0.4595 - val_loss: 0.0792 - val_accuracy: 0.4796 - 270ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.0733 - accuracy: 0.4707 - val_loss: 0.0708 - val_accuracy: 0.4781 - 270ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0674 - accuracy: 0.4763 - val_loss: 0.0671 - val_accuracy: 0.4818 - 272ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0639 - accuracy: 0.4796 - val_loss: 0.0647 - val_accuracy: 0.4622 - 273ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0614 - accuracy: 0.4865 - val_loss: 0.0626 - val_accuracy: 0.4705 - 277ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0594 - accuracy: 0.4841 - val_loss: 0.0618 - val_accuracy: 0.4879 - 271ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0579 - accuracy: 0.4840 - val_loss: 0.0608 - val_accuracy: 0.5023 - 269ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0367 - accuracy: 0.5267 - val_loss: 0.0554 - val_accuracy: 0.5136 - 268ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0368 - accuracy: 0.5259 - val_loss: 0.0555 - val_accuracy: 0.5113 - 267ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0367 - accuracy: 0.5241 - val_loss: 0.0565 - val_accuracy: 0.5083 - 269ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0365 - accuracy: 0.5266 - val_loss: 0.0565 - val_accuracy: 0.5151 - 271ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0364 - accuracy: 0.5244 - val_loss: 0.0577 - val_accuracy: 0.5401 - 266ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0362 - accuracy: 0.5248 - val_loss: 0.0565 - val_accuracy: 0.5121 - 271ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0360 - accuracy: 0.5282 - val_loss: 0.0567 - val_accuracy: 0.5129 - 266ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0358 - accuracy: 0.5265 - val_loss: 0.0564 - val_accuracy: 0.5045 - 269ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0294 - accuracy: 0.5355 - val_loss: 0.0662 - val_accuracy: 0.5295 - 263ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0295 - accuracy: 0.5355 - val_loss: 0.0658 - val_accuracy: 0.5106 - 269ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0295 - accuracy: 0.5349 - val_loss: 0.0663 - val_accuracy: 0.5219 - 267ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0292 - accuracy: 0.5370 - val_loss: 0.0654 - val_accuracy: 0.5083 - 269ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0292 - accuracy: 0.5360 - val_loss: 0.0666 - val_accuracy: 0.5151 - 267ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0291 - accuracy: 0.5348 - val_loss: 0.0661 - val_accuracy: 0.5166 - 270ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0290 - accuracy: 0.5350 - val_loss: 0.0659 - val_accuracy: 0.4962 - 266ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0289 - accuracy: 0.5342 - val_loss: 0.0669 - val_accuracy: 0.5015 - 282ms/epoch - 3

93/93 - 0s - loss: 0.0251 - accuracy: 0.5436 - val_loss: 0.0773 - val_accuracy: 0.5204 - 278ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0251 - accuracy: 0.5404 - val_loss: 0.0774 - val_accuracy: 0.5129 - 276ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0248 - accuracy: 0.5411 - val_loss: 0.0783 - val_accuracy: 0.5053 - 276ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0292 - accuracy: 0.5505
loss accuracy
104/104 [==============================] - 0s 1ms/step
0.6199767887778787


In [22]:
GE = ctx.docblocks[6]
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.3038 - accuracy: 0.1854 - val_loss: 0.1668 - val_accuracy: 0.2882 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1576 - accuracy: 0.3191 - val_loss: 0.1449 - val_accuracy: 0.3585 - 272ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1328 - accuracy: 0.3936 - val_loss: 0.1216 - val_accuracy: 0.4470 - 272ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.1133 - accuracy: 0.4291 - val_loss: 0.1062 - val_accuracy: 0.4539 - 274ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0996 - accuracy: 0.4439 - val_loss: 0.0944 - val_accuracy: 0.4849 - 274ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0893 - accuracy: 0.4576 - val_loss: 0.0866 - val_accuracy: 0.4652 - 274ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0823 - accuracy: 0.4574 - val_loss: 0.0809 - val_accuracy: 0.4614 - 271ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0774 - accuracy: 0.4587 - val_loss: 0.0770 - val_accuracy: 0.4561 - 271ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0519 - accuracy: 0.5017 - val_loss: 0.0557 - val_accuracy: 0.5083 - 273ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0517 - accuracy: 0.5022 - val_loss: 0.0558 - val_accuracy: 0.5038 - 274ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0517 - accuracy: 0.5024 - val_loss: 0.0558 - val_accuracy: 0.4992 - 274ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0515 - accuracy: 0.5029 - val_loss: 0.0554 - val_accuracy: 0.5068 - 267ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0514 - accuracy: 0.4993 - val_loss: 0.0558 - val_accuracy: 0.5030 - 269ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0513 - accuracy: 0.5016 - val_loss: 0.0551 - val_accuracy: 0.5091 - 271ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0512 - accuracy: 0.5026 - val_loss: 0.0550 - val_accuracy: 0.4909 - 267ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0510 - accuracy: 0.4986 - val_loss: 0.0555 - val_accuracy: 0.5151 - 268ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0460 - accuracy: 0.5066 - val_loss: 0.0532 - val_accuracy: 0.5030 - 271ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0459 - accuracy: 0.5099 - val_loss: 0.0534 - val_accuracy: 0.5106 - 264ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0458 - accuracy: 0.5095 - val_loss: 0.0537 - val_accuracy: 0.5008 - 272ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0458 - accuracy: 0.5058 - val_loss: 0.0533 - val_accuracy: 0.5008 - 272ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0457 - accuracy: 0.5080 - val_loss: 0.0535 - val_accuracy: 0.5091 - 270ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0456 - accuracy: 0.5098 - val_loss: 0.0534 - val_accuracy: 0.4909 - 272ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0456 - accuracy: 0.5083 - val_loss: 0.0532 - val_accuracy: 0.4871 - 274ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0455 - accuracy: 0.5097 - val_loss: 0.0536 - val_accuracy: 0.5015 - 277ms/epoch - 3

93/93 - 0s - loss: 0.0426 - accuracy: 0.5169 - val_loss: 0.0545 - val_accuracy: 0.5166 - 265ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0425 - accuracy: 0.5164 - val_loss: 0.0541 - val_accuracy: 0.5219 - 269ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0425 - accuracy: 0.5149 - val_loss: 0.0540 - val_accuracy: 0.5189 - 268ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0429 - accuracy: 0.5202
loss accuracy
104/104 [==============================] - 0s 2ms/step
0.6796123293398497


In [21]:
GE = ctx.docblocks[8]
GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.2976 - accuracy: 0.2944 - val_loss: 0.1149 - val_accuracy: 0.4342 - 1s/epoch - 14ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1010 - accuracy: 0.4312 - val_loss: 0.0942 - val_accuracy: 0.4455 - 267ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.0882 - accuracy: 0.4404 - val_loss: 0.0865 - val_accuracy: 0.4365 - 265ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0822 - accuracy: 0.4474 - val_loss: 0.0826 - val_accuracy: 0.4349 - 270ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0785 - accuracy: 0.4429 - val_loss: 0.0805 - val_accuracy: 0.4539 - 268ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0759 - accuracy: 0.4474 - val_loss: 0.0784 - val_accuracy: 0.4508 - 270ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0741 - accuracy: 0.4502 - val_loss: 0.0776 - val_accuracy: 0.4486 - 270ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0724 - accuracy: 0.4546 - val_loss: 0.0765 - val_accuracy: 0.4660 - 270ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0515 - accuracy: 0.4901 - val_loss: 0.0725 - val_accuracy: 0.4932 - 272ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0513 - accuracy: 0.4926 - val_loss: 0.0721 - val_accuracy: 0.4902 - 272ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0514 - accuracy: 0.4922 - val_loss: 0.0724 - val_accuracy: 0.4818 - 273ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0511 - accuracy: 0.4926 - val_loss: 0.0733 - val_accuracy: 0.4924 - 269ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0510 - accuracy: 0.4960 - val_loss: 0.0719 - val_accuracy: 0.4766 - 266ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0507 - accuracy: 0.4944 - val_loss: 0.0728 - val_accuracy: 0.4924 - 270ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0508 - accuracy: 0.4919 - val_loss: 0.0724 - val_accuracy: 0.4864 - 269ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0506 - accuracy: 0.4944 - val_loss: 0.0728 - val_accuracy: 0.4758 - 267ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0440 - accuracy: 0.4951 - val_loss: 0.0815 - val_accuracy: 0.4894 - 266ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0439 - accuracy: 0.4987 - val_loss: 0.0820 - val_accuracy: 0.4841 - 270ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0439 - accuracy: 0.5003 - val_loss: 0.0818 - val_accuracy: 0.4834 - 273ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0438 - accuracy: 0.4965 - val_loss: 0.0820 - val_accuracy: 0.4796 - 277ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0438 - accuracy: 0.5017 - val_loss: 0.0816 - val_accuracy: 0.4864 - 271ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0436 - accuracy: 0.4989 - val_loss: 0.0813 - val_accuracy: 0.4720 - 269ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0436 - accuracy: 0.4952 - val_loss: 0.0820 - val_accuracy: 0.4909 - 265ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0435 - accuracy: 0.4984 - val_loss: 0.0827 - val_accuracy: 0.4841 - 269ms/epoch - 3

93/93 - 0s - loss: 0.0394 - accuracy: 0.5032 - val_loss: 0.0909 - val_accuracy: 0.4773 - 274ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0393 - accuracy: 0.5085 - val_loss: 0.0914 - val_accuracy: 0.4811 - 276ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0393 - accuracy: 0.5087 - val_loss: 0.0914 - val_accuracy: 0.4697 - 273ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0431 - accuracy: 0.5040
loss accuracy
104/104 [==============================] - 0s 1ms/step
0.601665152891311


In [23]:
GE = ctx.docblocks[8]
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.3085 - accuracy: 0.1781 - val_loss: 0.1673 - val_accuracy: 0.2307 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1594 - accuracy: 0.3073 - val_loss: 0.1464 - val_accuracy: 0.3434 - 270ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1361 - accuracy: 0.3595 - val_loss: 0.1270 - val_accuracy: 0.4145 - 276ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.1206 - accuracy: 0.4144 - val_loss: 0.1150 - val_accuracy: 0.4569 - 274ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.1101 - accuracy: 0.4375 - val_loss: 0.1064 - val_accuracy: 0.4501 - 274ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.1023 - accuracy: 0.4381 - val_loss: 0.0998 - val_accuracy: 0.4554 - 274ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0965 - accuracy: 0.4412 - val_loss: 0.0957 - val_accuracy: 0.4592 - 273ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0921 - accuracy: 0.4412 - val_loss: 0.0913 - val_accuracy: 0.4584 - 272ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0676 - accuracy: 0.4655 - val_loss: 0.0713 - val_accuracy: 0.4713 - 270ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0675 - accuracy: 0.4640 - val_loss: 0.0710 - val_accuracy: 0.4622 - 269ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0674 - accuracy: 0.4651 - val_loss: 0.0715 - val_accuracy: 0.4682 - 277ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0674 - accuracy: 0.4654 - val_loss: 0.0709 - val_accuracy: 0.4728 - 278ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0672 - accuracy: 0.4635 - val_loss: 0.0710 - val_accuracy: 0.4796 - 274ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0670 - accuracy: 0.4676 - val_loss: 0.0708 - val_accuracy: 0.4660 - 280ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0670 - accuracy: 0.4654 - val_loss: 0.0707 - val_accuracy: 0.4735 - 274ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0668 - accuracy: 0.4657 - val_loss: 0.0716 - val_accuracy: 0.4720 - 269ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0623 - accuracy: 0.4732 - val_loss: 0.0694 - val_accuracy: 0.4690 - 288ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0622 - accuracy: 0.4760 - val_loss: 0.0691 - val_accuracy: 0.4705 - 273ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0621 - accuracy: 0.4741 - val_loss: 0.0689 - val_accuracy: 0.4478 - 271ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0621 - accuracy: 0.4751 - val_loss: 0.0691 - val_accuracy: 0.4644 - 273ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0620 - accuracy: 0.4719 - val_loss: 0.0695 - val_accuracy: 0.4743 - 279ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0619 - accuracy: 0.4769 - val_loss: 0.0695 - val_accuracy: 0.4713 - 275ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0619 - accuracy: 0.4765 - val_loss: 0.0692 - val_accuracy: 0.4644 - 271ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0618 - accuracy: 0.4743 - val_loss: 0.0693 - val_accuracy: 0.4652 - 274ms/epoch - 3

93/93 - 0s - loss: 0.0589 - accuracy: 0.4839 - val_loss: 0.0689 - val_accuracy: 0.4750 - 273ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0589 - accuracy: 0.4819 - val_loss: 0.0688 - val_accuracy: 0.4713 - 267ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0587 - accuracy: 0.4794 - val_loss: 0.0688 - val_accuracy: 0.4682 - 272ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0593 - accuracy: 0.4769
loss accuracy
104/104 [==============================] - 0s 2ms/step
0.6370781252252642


In [26]:
GE = ctx.docblocks[7]  # without norms works better everywhere, just check other dssms
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.3119 - accuracy: 0.1926 - val_loss: 0.1673 - val_accuracy: 0.2753 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1575 - accuracy: 0.3302 - val_loss: 0.1449 - val_accuracy: 0.3540 - 267ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1356 - accuracy: 0.4021 - val_loss: 0.1273 - val_accuracy: 0.4221 - 265ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.1201 - accuracy: 0.4439 - val_loss: 0.1151 - val_accuracy: 0.4652 - 277ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.1091 - accuracy: 0.4492 - val_loss: 0.1063 - val_accuracy: 0.4440 - 270ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.1010 - accuracy: 0.4485 - val_loss: 0.0987 - val_accuracy: 0.4455 - 270ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0943 - accuracy: 0.4506 - val_loss: 0.0928 - val_accuracy: 0.4342 - 267ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0891 - accuracy: 0.4524 - val_loss: 0.0886 - val_accuracy: 0.4713 - 265ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0602 - accuracy: 0.4825 - val_loss: 0.0638 - val_accuracy: 0.5038 - 270ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0599 - accuracy: 0.4841 - val_loss: 0.0634 - val_accuracy: 0.4962 - 277ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0597 - accuracy: 0.4847 - val_loss: 0.0635 - val_accuracy: 0.4977 - 281ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0596 - accuracy: 0.4857 - val_loss: 0.0631 - val_accuracy: 0.5008 - 279ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0595 - accuracy: 0.4840 - val_loss: 0.0626 - val_accuracy: 0.4909 - 267ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0593 - accuracy: 0.4840 - val_loss: 0.0628 - val_accuracy: 0.4788 - 268ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0592 - accuracy: 0.4859 - val_loss: 0.0625 - val_accuracy: 0.4871 - 268ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0591 - accuracy: 0.4846 - val_loss: 0.0624 - val_accuracy: 0.4720 - 272ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0537 - accuracy: 0.4921 - val_loss: 0.0615 - val_accuracy: 0.5030 - 268ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0535 - accuracy: 0.4929 - val_loss: 0.0607 - val_accuracy: 0.4796 - 273ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0535 - accuracy: 0.4920 - val_loss: 0.0613 - val_accuracy: 0.5030 - 278ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0535 - accuracy: 0.4923 - val_loss: 0.0607 - val_accuracy: 0.4864 - 270ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0532 - accuracy: 0.4931 - val_loss: 0.0606 - val_accuracy: 0.4887 - 274ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0533 - accuracy: 0.4926 - val_loss: 0.0607 - val_accuracy: 0.5000 - 279ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0531 - accuracy: 0.4911 - val_loss: 0.0608 - val_accuracy: 0.4939 - 270ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0531 - accuracy: 0.4909 - val_loss: 0.0604 - val_accuracy: 0.4788 - 277ms/epoch - 3

93/93 - 0s - loss: 0.0499 - accuracy: 0.4934 - val_loss: 0.0606 - val_accuracy: 0.4955 - 270ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0499 - accuracy: 0.4980 - val_loss: 0.0608 - val_accuracy: 0.4902 - 269ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0498 - accuracy: 0.4972 - val_loss: 0.0606 - val_accuracy: 0.4909 - 269ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0503 - accuracy: 0.4888
loss accuracy
104/104 [==============================] - 0s 2ms/step
0.6538276891137926


In [27]:
GE = ctx.docblocks[9]  # without norms works better everywhere, just check other dssms
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.2849 - accuracy: 0.1574 - val_loss: 0.1717 - val_accuracy: 0.2148 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1682 - accuracy: 0.2648 - val_loss: 0.1616 - val_accuracy: 0.3283 - 273ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1565 - accuracy: 0.2955 - val_loss: 0.1505 - val_accuracy: 0.3177 - 264ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.1475 - accuracy: 0.3067 - val_loss: 0.1436 - val_accuracy: 0.3638 - 270ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.1411 - accuracy: 0.3397 - val_loss: 0.1374 - val_accuracy: 0.3737 - 269ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.1356 - accuracy: 0.3601 - val_loss: 0.1329 - val_accuracy: 0.3949 - 271ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.1319 - accuracy: 0.3748 - val_loss: 0.1303 - val_accuracy: 0.3949 - 265ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.1293 - accuracy: 0.3830 - val_loss: 0.1282 - val_accuracy: 0.4123 - 270ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.1067 - accuracy: 0.4093 - val_loss: 0.1111 - val_accuracy: 0.3956 - 283ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.1065 - accuracy: 0.4116 - val_loss: 0.1107 - val_accuracy: 0.4175 - 275ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.1063 - accuracy: 0.4114 - val_loss: 0.1111 - val_accuracy: 0.4153 - 277ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.1061 - accuracy: 0.4101 - val_loss: 0.1111 - val_accuracy: 0.4274 - 276ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.1062 - accuracy: 0.4102 - val_loss: 0.1113 - val_accuracy: 0.4228 - 277ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.1062 - accuracy: 0.4106 - val_loss: 0.1109 - val_accuracy: 0.4349 - 278ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.1058 - accuracy: 0.4120 - val_loss: 0.1105 - val_accuracy: 0.4168 - 280ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.1057 - accuracy: 0.4106 - val_loss: 0.1107 - val_accuracy: 0.4281 - 282ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.1006 - accuracy: 0.4232 - val_loss: 0.1082 - val_accuracy: 0.3979 - 274ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.1006 - accuracy: 0.4232 - val_loss: 0.1074 - val_accuracy: 0.4357 - 268ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.1005 - accuracy: 0.4254 - val_loss: 0.1078 - val_accuracy: 0.4455 - 270ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.1004 - accuracy: 0.4249 - val_loss: 0.1074 - val_accuracy: 0.4206 - 271ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.1005 - accuracy: 0.4239 - val_loss: 0.1075 - val_accuracy: 0.4312 - 273ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.1002 - accuracy: 0.4250 - val_loss: 0.1073 - val_accuracy: 0.4070 - 277ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.1002 - accuracy: 0.4254 - val_loss: 0.1080 - val_accuracy: 0.4304 - 267ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.1002 - accuracy: 0.4243 - val_loss: 0.1092 - val_accuracy: 0.4236 - 273ms/epoch - 3

93/93 - 0s - loss: 0.0971 - accuracy: 0.4375 - val_loss: 0.1066 - val_accuracy: 0.4168 - 273ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0971 - accuracy: 0.4322 - val_loss: 0.1063 - val_accuracy: 0.4463 - 268ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0971 - accuracy: 0.4359 - val_loss: 0.1065 - val_accuracy: 0.4289 - 273ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0979 - accuracy: 0.4362
loss accuracy
104/104 [==============================] - 0s 2ms/step
0.5318173954413736


In [17]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [18]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

GE = ma.get_game_embs().numpy()
GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

/var/tmp/ipykernel_4168727/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6684629901446845 0.13094301466182667 4769
np.mean(results), mse, len(results) =  0.6609386973180077 0.14711578383534873 2088
Epoch 1/200
93/93 - 1s - loss: 0.3390 - accuracy: 0.2964 - val_loss: 0.1103 - val_accuracy: 0.4402 - 1s/epoch - 14ms/step
Epoch 2/200
93/93 - 0s - loss: 0.0825 - accuracy: 0.4769 - val_loss: 0.0701 - val_accuracy: 0.4652 - 273ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.0576 - accuracy: 0.4938 - val_loss: 0.0543 - val_accuracy: 0.4834 - 273ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.0457 - accuracy: 0.5060 - val_loss: 0.0458 - val_accuracy: 0.4728 - 270ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0388 - accuracy: 0.5125 - val_loss: 0.0410 - val_accuracy: 0.4811 - 274ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0343 - accuracy: 0.5230 - val_loss: 0.0377 - val_accuracy: 0.4894 - 270ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0312 - accuracy: 0.5264 - val_loss: 0.0356 - val_ac

Epoch 66/200
93/93 - 0s - loss: 0.0057 - accuracy: 0.5789 - val_loss: 0.0506 - val_accuracy: 0.5461 - 263ms/epoch - 3ms/step
Epoch 67/200
93/93 - 0s - loss: 0.0056 - accuracy: 0.5780 - val_loss: 0.0505 - val_accuracy: 0.5431 - 267ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0055 - accuracy: 0.5774 - val_loss: 0.0514 - val_accuracy: 0.5560 - 266ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0053 - accuracy: 0.5845 - val_loss: 0.0527 - val_accuracy: 0.5560 - 303ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0051 - accuracy: 0.5817 - val_loss: 0.0525 - val_accuracy: 0.5567 - 261ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0050 - accuracy: 0.5839 - val_loss: 0.0528 - val_accuracy: 0.5598 - 260ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0049 - accuracy: 0.5820 - val_loss: 0.0548 - val_accuracy: 0.5461 - 265ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0048 - accuracy: 0.5766 - val_loss: 0.0541 - val_accuracy: 0.5651 - 262ms/epoch - 3ms/step


Epoch 131/200
93/93 - 0s - loss: 4.8420e-04 - accuracy: 0.6057 - val_loss: 0.0997 - val_accuracy: 0.5590 - 271ms/epoch - 3ms/step
Epoch 132/200
93/93 - 0s - loss: 4.9370e-04 - accuracy: 0.6054 - val_loss: 0.1023 - val_accuracy: 0.5741 - 272ms/epoch - 3ms/step
Epoch 133/200
93/93 - 0s - loss: 6.4484e-04 - accuracy: 0.6049 - val_loss: 0.1036 - val_accuracy: 0.5613 - 287ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0015 - accuracy: 0.6051 - val_loss: 0.1000 - val_accuracy: 0.5575 - 273ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0027 - accuracy: 0.6085 - val_loss: 0.0976 - val_accuracy: 0.5809 - 270ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0019 - accuracy: 0.6055 - val_loss: 0.0990 - val_accuracy: 0.5847 - 266ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0010 - accuracy: 0.6117 - val_loss: 0.1000 - val_accuracy: 0.5802 - 268ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 5.8299e-04 - accuracy: 0.6125 - val_loss: 0.1002 - val_accuracy: 0.5832 -

Epoch 195/200
93/93 - 0s - loss: 8.7860e-05 - accuracy: 0.6122 - val_loss: 0.1277 - val_accuracy: 0.5734 - 268ms/epoch - 3ms/step
Epoch 196/200
93/93 - 0s - loss: 8.5379e-05 - accuracy: 0.6117 - val_loss: 0.1279 - val_accuracy: 0.5772 - 263ms/epoch - 3ms/step
Epoch 197/200
93/93 - 0s - loss: 8.2762e-05 - accuracy: 0.6115 - val_loss: 0.1284 - val_accuracy: 0.5741 - 265ms/epoch - 3ms/step
Epoch 198/200
93/93 - 0s - loss: 8.0058e-05 - accuracy: 0.6114 - val_loss: 0.1285 - val_accuracy: 0.5756 - 262ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 7.7719e-05 - accuracy: 0.6116 - val_loss: 0.1287 - val_accuracy: 0.5772 - 264ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 7.5673e-05 - accuracy: 0.6118 - val_loss: 0.1293 - val_accuracy: 0.5756 - 268ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0130 - accuracy: 0.6085
loss accuracy
104/104 [==============================] - 0s 1ms/step
0.6352053688565951


In [19]:
GE = ma.get_game_embs().numpy()
# GE = (GE - GE.mean(axis=0)) / GE.std(axis=0)

split = int(T.shape[0] * 0.8)
GE_train, GE_test = GE[idx][:split], GE[idx][split:]
T_train, T_test = T[idx][:split], T[idx][split:]

model = fit(GE_train, T_train)

best_th = sorted([
    (getQ(model, GE_train, T_train, th), th)
    for th in np.linspace(0, 1, 21)
])[-1][1]

print(getQ(model, GE_test, T_test, best_th))

Epoch 1/200
93/93 - 1s - loss: 0.3674 - accuracy: 0.1523 - val_loss: 0.1703 - val_accuracy: 0.2980 - 1s/epoch - 15ms/step
Epoch 2/200
93/93 - 0s - loss: 0.1606 - accuracy: 0.3150 - val_loss: 0.1475 - val_accuracy: 0.3669 - 268ms/epoch - 3ms/step
Epoch 3/200
93/93 - 0s - loss: 0.1335 - accuracy: 0.4238 - val_loss: 0.1192 - val_accuracy: 0.4614 - 263ms/epoch - 3ms/step
Epoch 4/200
93/93 - 0s - loss: 0.1069 - accuracy: 0.4608 - val_loss: 0.0968 - val_accuracy: 0.4697 - 270ms/epoch - 3ms/step
Epoch 5/200
93/93 - 0s - loss: 0.0887 - accuracy: 0.4677 - val_loss: 0.0827 - val_accuracy: 0.4652 - 269ms/epoch - 3ms/step
Epoch 6/200
93/93 - 0s - loss: 0.0767 - accuracy: 0.4735 - val_loss: 0.0730 - val_accuracy: 0.4735 - 285ms/epoch - 3ms/step
Epoch 7/200
93/93 - 0s - loss: 0.0677 - accuracy: 0.4818 - val_loss: 0.0653 - val_accuracy: 0.4841 - 269ms/epoch - 3ms/step
Epoch 8/200
93/93 - 0s - loss: 0.0608 - accuracy: 0.4911 - val_loss: 0.0597 - val_accuracy: 0.4818 - 270ms/epoch - 3ms/step
Epoch 9/20

Epoch 67/200
93/93 - 0s - loss: 0.0216 - accuracy: 0.5575 - val_loss: 0.0277 - val_accuracy: 0.5113 - 278ms/epoch - 3ms/step
Epoch 68/200
93/93 - 0s - loss: 0.0215 - accuracy: 0.5509 - val_loss: 0.0274 - val_accuracy: 0.5272 - 272ms/epoch - 3ms/step
Epoch 69/200
93/93 - 0s - loss: 0.0213 - accuracy: 0.5594 - val_loss: 0.0274 - val_accuracy: 0.5151 - 272ms/epoch - 3ms/step
Epoch 70/200
93/93 - 0s - loss: 0.0212 - accuracy: 0.5541 - val_loss: 0.0274 - val_accuracy: 0.5227 - 269ms/epoch - 3ms/step
Epoch 71/200
93/93 - 0s - loss: 0.0210 - accuracy: 0.5598 - val_loss: 0.0273 - val_accuracy: 0.5121 - 271ms/epoch - 3ms/step
Epoch 72/200
93/93 - 0s - loss: 0.0209 - accuracy: 0.5552 - val_loss: 0.0272 - val_accuracy: 0.5098 - 276ms/epoch - 3ms/step
Epoch 73/200
93/93 - 0s - loss: 0.0207 - accuracy: 0.5569 - val_loss: 0.0272 - val_accuracy: 0.5061 - 269ms/epoch - 3ms/step
Epoch 74/200
93/93 - 0s - loss: 0.0207 - accuracy: 0.5521 - val_loss: 0.0270 - val_accuracy: 0.5219 - 276ms/epoch - 3ms/step


Epoch 133/200
93/93 - 0s - loss: 0.0153 - accuracy: 0.5678 - val_loss: 0.0270 - val_accuracy: 0.5401 - 267ms/epoch - 3ms/step
Epoch 134/200
93/93 - 0s - loss: 0.0152 - accuracy: 0.5683 - val_loss: 0.0267 - val_accuracy: 0.5325 - 266ms/epoch - 3ms/step
Epoch 135/200
93/93 - 0s - loss: 0.0151 - accuracy: 0.5694 - val_loss: 0.0267 - val_accuracy: 0.5318 - 267ms/epoch - 3ms/step
Epoch 136/200
93/93 - 0s - loss: 0.0151 - accuracy: 0.5698 - val_loss: 0.0267 - val_accuracy: 0.5401 - 273ms/epoch - 3ms/step
Epoch 137/200
93/93 - 0s - loss: 0.0150 - accuracy: 0.5696 - val_loss: 0.0268 - val_accuracy: 0.5295 - 266ms/epoch - 3ms/step
Epoch 138/200
93/93 - 0s - loss: 0.0149 - accuracy: 0.5683 - val_loss: 0.0270 - val_accuracy: 0.5484 - 265ms/epoch - 3ms/step
Epoch 139/200
93/93 - 0s - loss: 0.0149 - accuracy: 0.5667 - val_loss: 0.0270 - val_accuracy: 0.5318 - 273ms/epoch - 3ms/step
Epoch 140/200
93/93 - 0s - loss: 0.0148 - accuracy: 0.5689 - val_loss: 0.0270 - val_accuracy: 0.5477 - 273ms/epoch - 3

93/93 - 0s - loss: 0.0114 - accuracy: 0.5750 - val_loss: 0.0297 - val_accuracy: 0.5499 - 265ms/epoch - 3ms/step
Epoch 199/200
93/93 - 0s - loss: 0.0114 - accuracy: 0.5746 - val_loss: 0.0295 - val_accuracy: 0.5560 - 266ms/epoch - 3ms/step
Epoch 200/200
93/93 - 0s - loss: 0.0113 - accuracy: 0.5753 - val_loss: 0.0296 - val_accuracy: 0.5484 - 271ms/epoch - 3ms/step
413/413 [==============================] - 1s 2ms/step - loss: 0.0127 - accuracy: 0.5706
loss accuracy
104/104 [==============================] - 0s 1ms/step
0.751599555959229


In [20]:
k = 1
(P * T)[k], np.clip(T + P, 0, 1)[k], T[k], P[k]

NameError: name 'P' is not defined

In [28]:
len(C)

30